In [24]:
import pandas as pd
import numpy as np

databases

In [25]:
orders = '../databases/orders.csv'
order_items = '../databases/order_items.csv'
products = '../databases/products.csv'
users = '../databases/users.csv'

read csv

In [26]:
orders_df = pd.read_csv(orders)
order_items_df = pd.read_csv(order_items)
users_df = pd.read_csv(users)

C:\Users\User\AppData\Local\Temp\ipykernel_984\1125601531.py:3: DtypeWarning: Columns (8) have mixed types. Specify dtype option on import or set low_memory=False.
  users_df = pd.read_csv(users)


filter valid orders

In [27]:
valid_orders = orders_df[
    ~orders_df["status"].isin(["Cancelled", "Returned"])
].copy()

Revenue per order

In [28]:
order_revenue = (
    order_items_df
    .groupby("order_id", as_index=False)["sale_price"]
    .sum()
    .rename(columns={"sale_price": "order_revenue"})
)

Join revenue into orders

In [29]:
merged = valid_orders.merge(order_revenue, on="order_id", how="left")

Final aggregation per user

In [30]:
merged["created_at"] = pd.to_datetime(
    merged["created_at"],
    format="ISO8601"
).dt.tz_localize(None)

customer_metrics = (
    merged.groupby("user_id")
    .agg(
        last_order_date=("created_at", "max"),
        total_orders=("order_id", "nunique"),
        total_revenue=("order_revenue", "sum")
    )
    .reset_index()
)

last_purchases = customer_metrics["days_since_last_order"] = (
    pd.Timestamp.today().normalize()
    - customer_metrics["last_order_date"]
).dt.days

customer_metrics

,user_id,last_order_date,total_orders,total_revenue,days_since_last_order
0,2,2026-05-24 17:24:33.124692,2,232.450001,4
1,5,2026-03-14 17:22:22.000000,1,258.000000,75
2,6,2026-02-04 19:05:50.000000,1,16.879999,113
3,7,2023-10-19 23:35:32.000000,1,43.570002,952
4,9,2026-02-11 10:40:59.000000,2,162.740000,106
...,...,...,...,...,...
66118,99996,2026-03-17 04:02:39.000000,1,158.000000,72
66119,99997,2025-02-17 08:47:30.000000,2,66.940002,465
66120,99998,2024-02-16 08:43:15.000000,2,439.879987,832
66121,99999,2022-06-13 00:25:06.000000,1,20.000000,1445


Days since last order

In [31]:
customer_metrics["days_since_last_order"] = (
    pd.Timestamp.today().normalize()
    - pd.to_datetime(customer_metrics["last_order_date"])
).dt.days

customer_metrics

,user_id,last_order_date,total_orders,total_revenue,days_since_last_order
0,2,2026-05-24 17:24:33.124692,2,232.450001,4
1,5,2026-03-14 17:22:22.000000,1,258.000000,75
2,6,2026-02-04 19:05:50.000000,1,16.879999,113
3,7,2023-10-19 23:35:32.000000,1,43.570002,952
4,9,2026-02-11 10:40:59.000000,2,162.740000,106
...,...,...,...,...,...
66118,99996,2026-03-17 04:02:39.000000,1,158.000000,72
66119,99997,2025-02-17 08:47:30.000000,2,66.940002,465
66120,99998,2024-02-16 08:43:15.000000,2,439.879987,832
66121,99999,2022-06-13 00:25:06.000000,1,20.000000,1445


In [33]:
customer_metrics["total_revenue"] = (
    customer_metrics["total_revenue"].round(2)
)

customer_metrics["churn_status"] = np.select(
    [
        customer_metrics["days_since_last_order"] <= 90,
        customer_metrics["days_since_last_order"] <= 180
    ],
    [
        "Active",
        "At Risk"
    ],
    default="Churned"
)

result = (
    customer_metrics[
        [
            "user_id",
            "last_order_date",
            "days_since_last_order",
            "total_orders",
            "total_revenue",
            "churn_status"
        ]
    ]
    .sort_values("days_since_last_order")
)

result

,user_id,last_order_date,days_since_last_order,total_orders,total_revenue,churn_status
36356,54989,2026-05-25 00:07:17.000000,3,1,155.00,Active
63041,95428,2026-05-25 00:12:43.616993,3,1,40.00,Active
49531,74979,2026-05-25 00:25:58.826170,3,1,230.53,Active
35229,53306,2026-05-25 00:19:47.582323,3,1,16.99,Active
62164,94093,2026-05-25 00:19:31.000000,3,1,21.00,Active
...,...,...,...,...,...,...
39613,60030,2019-01-29 08:04:38.000000,2676,1,328.10,Churned
17517,26483,2019-01-27 17:24:01.000000,2678,1,35.00,Churned
50617,76629,2019-01-26 21:08:13.000000,2679,1,65.84,Churned
54311,82251,2019-01-12 19:24:13.000000,2693,1,187.50,Churned
